## Biomass LA2 product Congo basin vs CCI Biomass
Objetive: Validate different areas of the Congo basin using the Biomass LA2 map against the information of the CCI biomass product

### 1. Packages and Libraries

In [ ]:
%pip install aiohttp requests

from shapely.geometry import Polygon, box
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import fiona
import os

fiona.supported_drivers["KML"] = "rw"  # Enable kml driver

from rasterio.windows import from_bounds
from rasterio.plot import show
from rasterio.plot import plotting_extent
from shapely.geometry import shape
from rasterio.enums import Resampling
import rioxarray  # adds .rio to xarray

## 2. Open AGB LA2 Congo map

In [ ]:
# Path to your GeoTIFF file
tif_path = r'Examples_and_Data/Workshops/BIOMASS-Coding2026/Biomass_AGBD_Congo_Basin/agbd_map_L2Afullextent_congo_basin.tif'

with rio.open(tif_path) as src:
    print("File:", src.name)
    print("Raster dimensions:", src.width, "x", src.height)
    print("Number of bands:", src.count)
    print("Coordinate reference system:", src.crs)
    print("Pixel resolution:", src.res)
    print("No-data value:", src.nodata)
    print("Band descriptions:", src.descriptions)
    print("Bounds:", src.bounds)

agb_bbox = [coord for coord in src.bounds]
agb_bbox

## 3. Query the ESA MAAP STAC catalog

In [ ]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

### 3.1.  Select a collection and narrow down the search
Using the bounding box of the AGB map from Congo get the CCI Biomass items

In [ ]:
search = catalog.search(
    collections = ["CCIBiomassV5.01"],
    datetime = ['2021-01-01T00:00:00Z','2021-12-31T00:00:00Z'],
    bbox = agb_bbox,
    method = "GET",
    max_items = 50,  
)

items = list(search.items())
print(f"Accessing {len(items)} items (limited by max_items).")
print(f"{search.matched()} items found that matched the query.")

Generate the gdf with the information of CCI biomass available

In [ ]:
features = []
for it in items:
    if it.geometry is None:
        raise ValueError(f"Item {it.id} has no geometry (footprint).")
    if "L4-AGB-MERGED" in it.id and "SD_MERGED" not in it.id:
        props = it.properties or {}
        features.append({
            "type": "Feature",
            "geometry": it.geometry,
            "properties": {
                "product_id": it.id,
                "start_datetime": props.get("start_datetime"),
                "end_datetime": props.get("end_datetime"),
                # "item": it
            },
        })


gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
gdf = gdf.reset_index()
# gdf = gdf[["index", "product_id", "start_datetime", "end_datetime", "geometry"]]
gdf = gdf[["product_id", "start_datetime", "end_datetime", "geometry"]]
gdf.head(len(items))

In [ ]:
# This is permanently valid, you don't need to change the two lines below.
CLIENT_ID="offline-token"
CLIENT_SECRET="p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"

**You need to create a .txt file and just paste your token inside**

In [ ]:
if pathlib.Path("GB_access_token.txt").exists(): # Put here your .txt file path
  with open("GB_access_token.txt","rt") as f:
    OFFLINE_TOKEN = f.read().strip().replace("\n","")
    url = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"
    data = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "refresh_token",
        "refresh_token": OFFLINE_TOKEN,
        "scope": "offline_access openid"
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    response_json = response.json()
    access_token = response_json.get('access_token')
    print("Access token retrieved successfully.")

    if not access_token:
        raise RuntimeError("Failed to retrieve access token from IAM response")

else:
    raise RuntimeError(f"No token file found in {os.getcwd()} !")

## 4. Apply the pipeline to each CCI product

In [ ]:
def crop_tif(path_tif, prod_id, bounds, path_folder_save):
    minx, miny, maxx, maxy = bounds
    # Replace these example coordinates with your desired zone
    left = minx
    bottom = miny
    right = maxx
    top = maxy
    with rio.open(path_tif) as src:
        # Convert map coordinates into a pixel window
        window = from_bounds(
            left, bottom, right, top,
            transform=src.transform
        )
        # Align the window to whole pixels
        window = window.round_offsets().round_lengths()
        # Crop the raster
        cropped = src.read(1, window=window, masked=True)
        # Update the spatial transform for the cropped raster
        cropped_transform = src.window_transform(window)
        print("Original size:", src.width, "x", src.height)
        print("Cropped size:", cropped.shape[1], "x", cropped.shape[0])
        # Calculate the correct map extent for the cropped zone
        extent = plotting_extent(cropped, cropped_transform)
        # Convert masked pixels to NaN for display
        cropped_data = cropped.filled(np.nan)
        valid = cropped_data[np.isfinite(cropped_data)]
        valid_pxs = valid.size
        if valid_pxs > 0:
            out_meta = src.meta
            cropped_data_nx = np.expand_dims(cropped_data, axis=0)
            cropped_data_nx.shape
            out_meta.update({"driver": "GTiff",
                 "height": cropped_data_nx.shape[1],
                 "width": cropped_data_nx.shape[2],
                 "transform": cropped_transform,
                 "count":1})
            path_tif_save = os.path.join(path_folder_save,f"agbd_map_L2A_cropped_{prod_id}.tif")
            with rio.open(os.path.join(path_folder_save,f"agbd_map_L2A_cropped_{prod_id}.tif"), "w", **out_meta) as dest:
                dest.write(cropped_data_nx)
            print(f'Tiff saved: {path_tif_save}')
            return path_tif_save
        else:
            print(f'Not valid pixels N: {valid_pxs}')
            return 0          

In [ ]:
path_folder_save = r'temp_tif'

In [ ]:
df_stats = pd.DataFrame()
for i, row in gdf.iterrows():
    prod_id = row['product_id']
    id_coords = prod_id.split('_')[0]
    cci_i_bounds = row['geometry'].bounds
    print(i, prod_id, cci_i_bounds)
    biomass_product = [id_ for id_ in items if prod_id in str(id_)][0]
    # Crop the Biomass LA2 map 
    path_tif_cropped = crop_tif(tif_path, prod_id, cci_i_bounds, path_folder_save)
    if path_tif_cropped == 0:
        continue
    else:
       # Open the agb map cropped with rioxarray
        agb_cropped = rioxarray.open_rasterio(path_tif_cropped, masked=True).squeeze().load()
        # cut the BIOMASS frame to the ALS tile
        agb_bounds = agb_cropped.rio.transform_bounds("EPSG:4326") 
        # Create an authenticated HTTPS filesystem for streaming access
        fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {access_token}"})
        
        # Open the remote raster file as a binary stream
        with fs.open(biomass_product.assets['enclosure_tif'].href, "rb") as f:
            cci_agb = (rioxarray.open_rasterio(f, masked=True).squeeze().rio.clip_box(*agb_bounds, crs="EPSG:4326").load())
        for name, da in [("Biomass LA2", agb_cropped), ("CCI biomass", cci_agb)]:
            print(f"{name:8s} CRS={da.rio.crs}  res={da.rio.resolution()}  shape={da.shape}  "
                  f"range=({float(da.min()):.1f}, {float(da.max()):.1f})")
        cci_on_agb_mean = cci_agb.rio.reproject_match(agb_cropped, resampling=Resampling.average)
        cci_on_agb_max = cci_agb.rio.reproject_match(agb_cropped, resampling=Resampling.max)
        valid = agb_cropped.notnull().astype("float32").rio.write_nodata(np.nan)
        coverage = valid.rio.reproject_match(agb_cropped, resampling=Resampling.average)
        
        MIN_COVER = 0.8
        cmp = pd.DataFrame({
            f"{id_coords}_CCI_biomass_mean": cci_on_agb_mean.values.ravel(),
            f"{id_coords}_CCI_biomass_max": cci_on_agb_max.values.ravel(),
            f"{id_coords}_Biomass_LA2": agb_cropped.values.ravel(),
            f"{id_coords}_cover": coverage.values.ravel(),
        }).dropna()
        cmp = cmp[cmp[f"{id_coords}_cover"] >= MIN_COVER]
        print(f"Matched BIOMASS pixels with >={MIN_COVER:.0%} CCI biomass coverage: {len(cmp)}")
        
        def stats(y_ref, y_est):
            d = y_est - y_ref
            r = np.corrcoef(y_ref, y_est)[0, 1]
            return {"n": len(d), "bias": d.mean(), "RMSE": np.sqrt((d**2).mean()),
                    "rel_RMSE_%": 100 * np.sqrt((d**2).mean()) / y_ref.mean(), "R2": r**2}
        
        
        df_stats_ = pd.DataFrame({k: stats(cmp[k], cmp[f"{id_coords}_Biomass_LA2"]) for k in [f"{id_coords}_CCI_biomass_mean",f"{id_coords}_CCI_biomass_max"]}).T.round(2)
        df_stats = pd.concat([df_stats,df_stats_], axis=0)
        # Plots
        # if df_stats_.loc[0,'R2'] > 0.5:
        #     fig, ax = plt.subplots(1, 3, figsize=(16, 5))
        #     vmax = float(np.nanpercentile(agb_cropped, 99))
        #     cci_on_agb_mean.plot(ax=ax[0], vmin=0, vmax=vmax, cmap="viridis"); ax[0].set_title(f"CCI biomass {id_coords}")
        #     agb_cropped.plot(ax=ax[1], vmin=0, vmax=vmax, cmap="viridis"); ax[1].set_title("BIOMASS L2A (mean, on CCI biomass grid)")
        #     (cci_on_agb_mean - agb_cropped).plot(ax=ax[2], cmap="RdBu_r", center=0); ax[2].set_title("BIOMASS − ALS")
        #     for a in ax:
        #         a.set_aspect("equal")
        #     plt.tight_layout()
        #     plt.show()
    # break

In [ ]:
# Select the elements with enough pixels to validate
df = df_stats[(~df_stats['R2'].isna()) & (df_stats['R2'] > 0.01)]
df

In [ ]:
# Save the df of the validation
path_save_csv = r'Validation_stats.csv'
df.to_csv(path_save_csv)

In [ ]:
# Select the mean idx for the mean biomass la2 values
idxs_mean = [idx_ for idx_ in df.index.values if 'mean' in idx_]

df_means = df.loc[idxs_mean]

df_means

In [ ]:
df_means['R2'].mean()